# Team Synthesis

In [1]:
from typing import Any

from kirin.ir import Method
import matplotlib.pyplot as plt
import numpy as np

from bloqade import squin, tsim
from bloqade.cirq_utils import emit_circuit, load_circuit, noise
from bloqade.pyqrack import StackMemorySimulator
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList
import matplotlib.pyplot as plt

# this will help us have return types for our methods that have more intuitive names
Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

# this function will help us visualize some circuits
def show_circuit(squin_kernel):
    @squin.kernel
    def _to_visualize():
        _ = squin_kernel()

    return tsim.Circuit(_to_visualize).diagram(height=400)

## Part 1: Learn the language of Clifford+$T$


Learn how to build and simulate simple circuits using Bloqade Squin and Bloqade PyQrack.

Build a few small 1-qubit and 2-qubit examples. Confirm that you understand how $H, S, $ and $CNOT$ act on simple input states. Use this part to get comfortable creating, simulating, and visualizing circuits constructed from the Clifford+$T := {H, S, CNOT, T}$ gateset.

Goal: Build intuition for Clifford+T circuits and the simulation workflow.

In [2]:
@squin.kernel
def bell_state() -> Measurement:
    qubits = squin.qalloc(2)
    squin.h(qubits[0])
    squin.cx(qubits[0], qubits[1])
    bits = squin.broadcast.measure(qubits)
    return bits

In [3]:
show_circuit(bell_state)

In [4]:
pyqrack_target = StackMemorySimulator(min_qubits=2)
task = pyqrack_target.task(bell_state)

single_shot = task.run()
single_shot

IList([<Measurement.One: 1>, <Measurement.One: 1>])

In [5]:
batch_results = task.batch_run(shots=2000)
batch_results

{(<Measurement.Zero: 0>, <Measurement.Zero: 0>): 0.508,
 (<Measurement.One: 1>, <Measurement.One: 1>): 0.492}

## Part 2: Synthesize the rotation family

Focus on the family

$$
R_z\left(\pi / 2^n\right), \qquad n \in 0,1,2,3,4,5
$$

How can we implement these “dyadic” Z-rotations using only Clifford+$T$?

Try synthesizing these rotations as well as you can using only our chosen gate set for one qubit (yes, only 1 qubit), and different values of \(n\). Some implementations may be exact while others may involve approximations, that is okay. It is up to you to explore different synthesis strategies and compare the circuits you find.

We suggest spending time on finding ways to visualize how your approximations act on different initial states, and reflecting on what you tried, how you judged quality, and what changed as the target angle became smaller.

**Goal**: Explore how small Z-rotations can be built from Clifford+ \(T\) and explain the synthesis strategies you explored.

**Distance metric**: When comparing a target gate \(U\) to an implementation \(V\), use the following global-phase-invariant distance:

$$
d(U,V) = \sqrt{1 - \frac{|\mathrm{Tr}(U^\dagger V)|}{2}}
$$

**Interpretation**
- \(d = 0\) means exact agreement up to global phase,
- larger \(d\) means a worse approximation.

In [30]:
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]])

gates = {
    "H": H,
    "T": T,
}

def rz(theta):
    return np.array([[1, 0], [0, np.exp(1j * theta)]])

def distance(U, V):
    return np.sqrt(1 - abs(np.trace(U.conj().T @ V)) / 2)

In [38]:
from itertools import product

def brute_force(n, max_depth):
    best_seq = None
    best_d = float("inf")
    best_U = None

    U = rz(np.pi / (2**n))

    for seq in product(gates.keys(), repeat=max_depth):
        V = np.eye(2)
        for gate in seq: V = gates[gate] @ V
        d = distance(U, V)
        
        if d < best_d:
            best_d = d
            best_seq = seq
            best_U = V

    return best_seq, best_d, best_U

In [40]:
for max_depth in range(1, 10):
    seq, d, U = brute_force(5, max_depth)
    print(f"Depth {max_depth}: best sequence {seq} with distance {d:.4f}")

Depth 1: best sequence ('T',) with distance 0.2418
Depth 2: best sequence ('H', 'H') with distance 0.0347
Depth 3: best sequence ('H', 'H', 'T') with distance 0.2418
Depth 4: best sequence ('H', 'H', 'H', 'H') with distance 0.0347
Depth 5: best sequence ('H', 'H', 'H', 'H', 'T') with distance 0.2418
Depth 6: best sequence ('H', 'H', 'H', 'H', 'H', 'H') with distance 0.0347
Depth 7: best sequence ('H', 'H', 'T', 'H', 'H', 'H', 'H') with distance 0.2418
Depth 8: best sequence ('T', 'T', 'T', 'T', 'T', 'T', 'T', 'T') with distance 0.0347
Depth 9: best sequence ('H', 'T', 'T', 'H', 'T', 'T', 'H', 'T', 'T') with distance 0.0347


In [6]:
def z_rotation(n):
    @squin.kernel
    def _exact_rotation():
        qubits = squin.qalloc(1)
        squin.rz(np.pi/(2**n), qubits[0])
        return qubits[0]
    return _exact_rotation

In [7]:
show_circuit(z_rotation(2))